In [18]:
import os
import bidict
import pickle
import duckdb

import jsonlines
import pandas as pd
import polars as pl

from orqa.utils import sanitize_string, is_num

In [2]:
tables_path     = '../data/datasets/CAN/tables/tables_from10000_to15000'
metadata_path   = '../data/datasets/CAN/metadata/metadata_from10000_to15000'

db_path         = '../data/datasets/CAN/database/CAN.db'
valdict_path    = '../data/datasets/CAN/database/values_dict.pickle'

In [3]:
table_ids = list(sorted(os.listdir(tables_path), reverse=True))
len(table_ids)

3689

In [29]:
with jsonlines.open(metadata_path) as fr:
    metadata = {rsc['id']: md for md in fr.iter() for rsc in md['resources'] if rsc['format'] == 'CSV'}

In [4]:
with open(valdict_path, 'rb') as fr:
    values = pickle.load(fr)

In [30]:
type(values), len(values.values()), len(set(values.values())), len(metadata)

(bidict.bidict, 5961157, 5961157, 4203)

In [6]:
from itertools import chain

def get_table_values(table_id):
    df = pl.read_parquet(f'{tables_path}/{table_id}')
    df = df[[s.name for s in df if not (s.null_count() == df.height)]]

    return set(
        filter(
            lambda v: v not in values and not is_num(v), 
            map(
                lambda s: sanitize_string(str(s)), 
                chain(*(
                    df.select(col).drop_nans().drop_nulls().unique().get_column(col).to_list() 
                    for col in df.columns
                    )
                )                
            )
        )
    )

In [7]:
con = duckdb.connect(db_path, read_only=True)

In [195]:
i_tab_q = 5
qdf = pl.read_parquet(f"{tables_path}/{table_ids[i_tab_q]}")
qdf

Sample No.  No. déchantillon,Region - Région,Commodity,Denrée,Function,Fonction,Origin,Origine,Product,Produit,Date Sampled  Date déchantillonage,Component,Composant,Amount,Report Unit - Unité de rapport
str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str
"""CFP2020-0001""","""ATLANTIC""","""Dairy""","""Laitier""","""Domestic""","""domestique""","""CANADA""","""CANADA""","""Yogurt""","""Yogourt""","""19-Oct-20""","""19-Nortestosterone""","""19-Nortestostérone""",0.0,"""µg/g"""
"""CFP2020-0001""","""ATLANTIC""","""Dairy""","""Laitier""","""Domestic""","""domestique""","""CANADA""","""CANADA""","""Yogurt""","""Yogourt""","""19-Oct-20""","""2,3,4,5 TetraChlorophenol""","""2,3,4,5 Tétrachlorophénol""",0.0,"""µg/g"""
"""CFP2020-0001""","""ATLANTIC""","""Dairy""","""Laitier""","""Domestic""","""domestique""","""CANADA""","""CANADA""","""Yogurt""","""Yogourt""","""19-Oct-20""","""2,3,4,6 TetraChlorophenol""","""2,3,4,6 Tétrachlorophénol""",0.0,"""µg/g"""
"""CFP2020-0001""","""ATLANTIC""","""Dairy""","""Laitier""","""Domestic""","""domestique""","""CANADA""","""CANADA""","""Yogurt""","""Yogourt""","""19-Oct-20""","""2,3,5,6 TetraChlorophenol""","""2,3,5,6 Tétrachlorophénol""",0.0,"""µg/g"""
"""CFP2020-0001""","""ATLANTIC""","""Dairy""","""Laitier""","""Domestic""","""domestique""","""CANADA""","""CANADA""","""Yogurt""","""Yogourt""","""19-Oct-20""","""20-Dihydroprednisolone""","""20-Dihydroprednisolone""",0.0,"""µg/g"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CFP2020-0175""","""ATLANTIC""","""IMFD""","""DAIM""","""Import""","""importé""","""UNKNOWN""","""INCONNU""","""Cereal - Children""","""Céréales - Enfants""","""29-Oct-20""","""Vinclozolin""","""Vinclozoline""",0.0,"""µg/g"""
"""CFP2020-0175""","""ATLANTIC""","""IMFD""","""DAIM""","""Import""","""importé""","""UNKNOWN""","""INCONNU""","""Cereal - Children""","""Céréales - Enfants""","""29-Oct-20""","""Zengxiaoan""","""zengxiaoan""",0.0,"""µg/g"""
"""CFP2020-0175""","""ATLANTIC""","""IMFD""","""DAIM""","""Import""","""importé""","""UNKNOWN""","""INCONNU""","""Cereal - Children""","""Céréales - Enfants""","""29-Oct-20""","""Zinc""","""Zinc""",22.3,"""µg/g"""


In [197]:
bad_columns_tokens = {'id', 'date'}

for i_col in range(len(qdf.columns)):
    if 'id' in qdf.columns[i_col].lower():
        continue
    print(i_col, qdf.columns[i_col])
    if any(tok in qdf.columns[i_col].lower() for tok in bad_columns_tokens):
        continue
    qcol = map(lambda v: sanitize_string(str(v)), set(qdf.to_series(i_col)))
    q = list(map(lambda v: str(values[v]), filter(lambda v: v in values, qcol)))
    if not q: continue
    res = con.sql(f"""
            SELECT TableId, ColumnId, COUNT(DISTINCT CellValue) AS intersec FROM AllTables
            WHERE CellValue IN ({','.join(q)})
            AND TableId <> {i_tab_q}
            GROUP BY TableId, ColumnId
            ORDER BY COUNT(DISTINCT CellValue) DESC
            LIMIT 10;
    """)
    print(res)

0 Sample No.  No. déchantillon
┌─────────┬──────────┬──────────┐
│ TableId │ ColumnId │ intersec │
│  int32  │  int32   │  int64   │
├─────────┴──────────┴──────────┤
│            0 rows             │
└───────────────────────────────┘

1 Region - Région
┌─────────┬──────────┬──────────┐
│ TableId │ ColumnId │ intersec │
│  int32  │  int32   │  int64   │
├─────────┼──────────┼──────────┤
│    2953 │        0 │        1 │
│    1086 │        0 │        1 │
│     987 │        0 │        1 │
│     685 │        1 │        1 │
│    1152 │        0 │        1 │
│    1026 │        0 │        1 │
│    3239 │        0 │        1 │
│     998 │        0 │        1 │
│     959 │        0 │        1 │
│    3515 │        0 │        1 │
├─────────┴──────────┴──────────┤
│ 10 rows             3 columns │
└───────────────────────────────┘

2 Commodity
┌─────────┬──────────┬──────────┐
│ TableId │ ColumnId │ intersec │
│  int32  │  int32   │  int64   │
├─────────┼──────────┼──────────┤
│    2880 │      

In [198]:
i_tab_res = 269
r_df = pl.read_parquet(f"{tables_path}/{table_ids[i_tab_res]}")
r_df

Sample No.  No. déchantillon,Region - Région,Function,Fonction,Origin,Origine,Product,Produit,Date Sampled  Date déchantillonage,Type of Test,Type danalyse,Component,Composant,Result - Résultat,Report Unit - Unité de rapport,Plan Code - Code du régime
str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""C2020BPHA00001""","""WEST""","""Import""","""importé""","""CHINA""","""CHINE""","""Peach""","""Pêche""","""17-Nov-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol A""","""Bisphénol A""",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00001""","""WEST""","""Import""","""importé""","""CHINA""","""CHINE""","""Peach""","""Pêche""","""17-Nov-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol A Diglycidyl Ether""","""Éther diglycidylique de bisphé…",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00001""","""WEST""","""Import""","""importé""","""CHINA""","""CHINE""","""Peach""","""Pêche""","""17-Nov-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol F""","""Bisphénol F""",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00001""","""WEST""","""Import""","""importé""","""CHINA""","""CHINE""","""Peach""","""Pêche""","""17-Nov-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol S""","""Bisphénol S""",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00001D""","""WEST""","""Import""","""importé""","""UNITED STATES""","""ÉTATS-UNIS""","""Pear""","""Poire""","""08-Feb-21""","""BPA""","""BISPHÉNOL A""","""Bisphenol A""","""Bisphénol A""",0.0,"""µg/g""","""2020_SB451"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""C2020BPHA00399""","""WEST""","""Unknown""","""inconnu""","""CANADA""","""CANADA""","""Meat - Canned""","""Produit de viande - en conserv…","""15-Jul-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol S""","""Bisphénol S""",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00400""","""WEST""","""Import""","""importé""","""THAILAND""","""THAÏLANDE""","""Fish/Seafood - Canned""","""Poisson/Fruits de mer - En con…","""05-Oct-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol A""","""Bisphénol A""",0.0,"""µg/g""","""2020_SB451"""
"""C2020BPHA00400""","""WEST""","""Import""","""importé""","""THAILAND""","""THAÏLANDE""","""Fish/Seafood - Canned""","""Poisson/Fruits de mer - En con…","""05-Oct-20""","""BPA""","""BISPHÉNOL A""","""Bisphenol A Diglycidyl Ether""","""Éther diglycidylique de bisphé…",0.0,"""µg/g""","""2020_SB451"""


In [183]:
r_df.to_series(6).is_null().sum() / r_df.shape[0]

0.8715203426124197

In [134]:
def get_resource_metadata(rsc_id):
    return next(filter(lambda r: r['id'] == table_ids[rsc_id].removesuffix('.parquet'), metadata[table_ids[rsc_id].removesuffix('.parquet')]['resources']))

In [199]:
print(get_resource_metadata(i_tab_q)['name'])
print(get_resource_metadata(i_tab_res)['name'])
print('-' * 20)
print(metadata[table_ids[i_tab_q].removesuffix('.parquet')]['title'])
print(metadata[table_ids[i_tab_res].removesuffix('.parquet')]['title'])

Children's Food Project - 2020 Data
Open Data for Chemistry Targeted Surveys - 2020-21 Bisphenol A and BPA Alternatives in Selected Canned Foods
--------------------
Children's Food Project - 2020 Data
Open Data for Chemistry Targeted Surveys - 2020-21 Bisphenol A and BPA Alternatives in Selected Canned Foods


In [201]:
print(metadata[table_ids[i_tab_q].removesuffix('.parquet')]['notes'])
print('-' * 10)
print(metadata[table_ids[i_tab_res].removesuffix('.parquet')]['notes'])

The main objectives of the 2020 CFP were to:

collect data and assess the compliance of infant foods to Canadian standards for residues of pesticides, veterinary drugs and metals/elements
collect data on aflatoxin M1 levels in yogurt/yogurt beverages and pudding
Previous years have analyzed pesticides, metals/elements, veterinary drug residues, aflatoxin M1 and environmental contaminants.

In the 2020 CFP, a total of 175 samples of children's foods were purchased in Halifax, Nova Scotia in October, 2020. These samples included:

children's cereals
yogurt/yogurt beverages
pudding
fruit snacks
granola/cereal bars
----------
​This dataset file and the results containe support and correlate with the CFIA Food Safety Bulletin Report - Bisphenol A and BPA Alternatives in Selected Canned Foods - April 1, 2020 to March 31, 2021.


In [203]:
metadata[table_ids[i_tab_q].removesuffix('.parquet')]

{'association_type': [],
 'audience': [],
 'author': None,
 'author_email': None,
 'collection': 'primary',
 'contributor': {'fr': '', 'en': ''},
 'creator': '',
 'creator_user_id': '2dc5a12f-2a6f-4822-8514-165ea6bb05d9',
 'data_series_issue_identification': {'fr': 'CFIA-ACIA-1264',
  'en': 'CFIA-ACIA-1264'},
 'data_series_name': {'fr': '', 'en': ''},
 'date_published': '2022-08-18 00:00:00',
 'digital_object_identifier': '',
 'display_flags': [],
 'frequency': 'not_planned',
 'geographic_region': [],
 'id': 'b1974f49-da37-4783-8705-97bfe6720a2c',
 'imso_approval': 'true',
 'isopen': False,
 'jurisdiction': 'federal',
 'keywords': {'fr': ['enfants',
   'bébés',
   'chimie',
   'salubrité alimentaire',
   'résidus chimiques'],
  'en': ['food safety',
   'chemistry',
   'children',
   'infant',
   'chemical residues']},
 'license_id': 'ca-ogl-lgo',
 'license_title': 'Open Government Licence - Canada',
 'license_url': 'https://open.canada.ca/en/open-government-licence-canada',
 'maintaine

In [ ]:
r_col = set(qdf.to_series(1))
s_col = set(r_df.to_series(1))

intersection = r_col & s_col
union = r_col | s_col
jaccard = round(len(intersection) / len(union), 3)
overlap = round(len(intersection) / min(len(r_col), len(s_col)), 3)
print(f'{len(intersection)=}, {len(union)=}, {jaccard=}, {overlap=}')

len(intersection)=1, len(union)=12, jaccard=0.083, overlap=1.0


In [142]:
intersection

{'Canada'}

In [ ]:
qdf.select(pl.nth(i_col)).unique().drop_nulls().join(
    r_df.select(pl.nth(0)).unique().drop_nulls(),
    left_on='Unnamed: 0', right_on='Unnamed: 0'
)

Unnamed: 0
str
"""Symbols: """
"""Coefficient of variation (CV)"""
"""Air transportation"""
"""Other"""
"""Maritime transportation"""
…
"""Components may not add up to t…"
"""Rail transportation"""
"""Totals and sub-totals have bee…"


In [ ]:
r_df.select(pl.nth(0)).unique().drop_nulls()

Unnamed: 0
str
"""Other"""
"""Source: """
""" not applicable"""
"""F = Greater than 25.00% too …"
"""All sizes"""
…
"""Table 43b"""
"""100 employees and more"""
"""Air transportation"""


In [ ]:
rcol = r_df.get_columns()[1].drop_nulls().to_list()
set(rcol).intersection(qcol)

set()

In [ ]:
# con.sql("SELECT DISTINCT(TableID) FROM OrqaIndex ORDER BY TableID LIMIT 5")
con.sql("SELECT * FROM AllTables WHERE TableId = 0 LIMIT 100")

┌─────────┬──────────┬───────┬───────────┐
│ TableId │ ColumnId │ RowId │ CellValue │
│  int32  │  int32   │ int32 │   int32   │
├─────────┼──────────┼───────┼───────────┤
│       0 │        0 │     0 │        48 │
│       0 │        0 │     1 │        48 │
│       0 │        0 │     2 │        48 │
│       0 │        0 │     3 │        48 │
│       0 │        0 │     4 │        48 │
│       0 │        0 │     5 │        48 │
│       0 │        0 │     6 │        48 │
│       0 │        0 │     7 │        48 │
│       0 │        0 │     8 │        48 │
│       0 │        0 │     9 │        48 │
│       · │        · │     · │         · │
│       · │        · │     · │         · │
│       · │        · │     · │         · │
│       0 │        0 │    30 │        48 │
│       0 │        0 │    31 │        48 │
│       0 │        0 │    32 │        48 │
│       0 │        0 │    33 │        48 │
│       0 │        0 │    34 │        48 │
│       0 │        0 │    35 │        48 │
│       0 │

In [41]:
pl.read_parquet(f"{tables_path}/{table_ids[0]}")

Category,Average Daily Count - Nova Scotia,Average Daily Count,Year
str,str,i64,str
"""Adult""","""Remand - adult""",227,"""2014-2015"""
"""Adult""","""Sentenced provincial custody -…",251,"""2014-2015"""
"""Adult""","""Other - adult""",22,"""2014-2015"""
"""Adult""","""Total count - adults""",500,"""2014-2015"""
"""Adult""","""Remand - adult""",223,"""2015-2016"""
…,…,…,…
"""Youth""","""Total count - youth""",7,"""2022-2023"""
"""Youth""","""Remand - youth""",6,"""2023-2024"""
"""Youth""","""Sentenced provincial custody -…",6,"""2023-2024"""
